# Advanced OCR — Colab runner

This notebook is **only a runner**. All code lives in the GitHub repo
`greencube-dev-boop/advanced-ocr` (synced below with `git pull` — push updates to
GitHub and just rerun this notebook, never re-upload it). All data lives in your
Google Drive under **`OCR Engine/`**.

Pipeline: raw PDF → page-by-page → native text / Tesseract → nemotron-3.5-parse
escalation on tables & suspicious pages → LLM refinement → one Markdown per book
in `OCR Engine/refined/`.

**Rerunning this notebook resumes where it stopped** (state: `OCR Engine/config/books.json`).

Run the cells top to bottom.

In [ ]:
# 1) Dependencies (idempotent). torch is preinstalled on Colab.
!pip install -q PyMuPDF pytesseract Pillow requests tqdm transformers accelerate bitsandbytes
!apt-get install -qq tesseract-ocr >/dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq tesseract-ocr)
print("deps ok")

In [ ]:
# 2) Mount Google Drive and set the OCR Engine root
import os
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/OCR Engine"
os.makedirs(DRIVE_ROOT, exist_ok=True)   # creates raw/ config/ refined/ temp/ on first use
print("Drive root:", DRIVE_ROOT)
for sub in ("raw", "config", "refined", "temp"):
    os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)
print("raw/ is where you drop PDFs:", os.path.join(DRIVE_ROOT, "raw"))

In [ ]:
# 3) Sync the codebase from GitHub (clone once, then git pull on every run)
import os, subprocess, sys

REPO_URL = "https://github.com/greencube-dev-boop/advanced-ocr.git"
GITHUB_TOKEN = ""  # repo is private: paste a GitHub PAT with repo-read here (or make the repo public)
WORKDIR = "/content/advanced-ocr"

url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
if os.path.isdir(os.path.join(WORKDIR, ".git")):
    subprocess.run(["git", "-C", WORKDIR, "pull", "--ff-only", url], check=False)
else:
    subprocess.run(["git", "clone", url, WORKDIR], check=True)

os.environ["PYTHONPATH"] = os.path.join(WORKDIR, "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, os.path.join(WORKDIR, "src"))
import ocr_pipeline
print("pipeline", ocr_pipeline.__version__, "from", WORKDIR)

In [ ]:
# 4) API keys — upload api.txt (line 1 = NVIDIA key; optional extra KEY=value lines).
#    Used only in this runtime; never written to Drive or git.
from google.colab import files
import shutil

uploaded = files.upload()          # choose your api.txt
shutil.move(next(iter(uploaded)), "/content/api.txt")
API_FILE = "/content/api.txt"
print("api.txt loaded")

In [ ]:
# 5) Sanity check: deps, keys, drive, tesseract
!python -m ocr_pipeline doctor --drive-root "{DRIVE_ROOT}" --api-file "{API_FILE}"
!python -m ocr_pipeline init-config --drive-root "{DRIVE_ROOT}" --api-file "{API_FILE}"

In [ ]:
# 6) Run / resume the pipeline.
#    Drop PDFs into Drive -> "OCR Engine/raw/" first.
#    Finished pages are skipped automatically; interrupt anytime and rerun to resume.
!python -m ocr_pipeline run --drive-root "{DRIVE_ROOT}" --api-file "{API_FILE}"

In [ ]:
# 7) Inspect resume state
!python -m ocr_pipeline status --drive-root "{DRIVE_ROOT}" --api-file "{API_FILE}"

## Outputs

- Combined book Markdown: `Drive/OCR Engine/refined/<book>.md`
- Per-page trail (native / ocr / nemotron / refined / meta): `Drive/OCR Engine/temp/<book>/`
- Resume state: `Drive/OCR Engine/config/books.json` — settings: `Drive/OCR Engine/config/pipeline.json`

## Tips

- Settings (models, dpi, tesseract language, suspicion thresholds, refiner backend) are edited in `Drive/OCR Engine/config/pipeline.json` — no code changes needed.
- Default refiner is **keyless Gemma 3 (4-bit) on the Colab GPU**. To use Gemini instead, add `GEMINI_API_KEY=...` to api.txt and set `refiner.backend = "gemini_api"` in the Drive config.
- To process a single book: `!python -m ocr_pipeline run-book mybook.pdf --drive-root "..." --api-file "..."`
